# 03 — Train Models

Trains and compares all four classifiers specified in the dissertation Research Method:
1. **Logistic Regression** (interpretable baseline)
2. **Random Forest** (ensemble, per Leckey et al. 2024)
3. **XGBoost** (gradient boosting)
4. **CNN + BiLSTM** (temporal model on per-frame sequences)

Models 1-3 train on the per-rep aggregated feature table (mean/std/range). Model 4 needs the raw per-frame sequence: if `SEQ_DATA_PATH` (from `02_extract_real_dataset.ipynb`) is available it trains on **real** per-frame sequences; otherwise it falls back to a reconstructed pseudo-sequence approximation (only appropriate for the synthetic dataset).

All four models are trained with stratified 5-fold CV, evaluated on a held-out 20% stratified test set, and saved to `models/`. Run `04_evaluate.ipynb` afterwards for the full metrics/plots/McNemar comparison.

In [ ]:
import sys
sys.path.append('.')

import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score
from xgboost import XGBClassifier

from biomechanics import FEATURE_COLUMNS

In [ ]:
# ---- CONFIG -- point these at real data once you've run 02_extract_real_dataset.ipynb ----
DATA_PATH = "../data/processed/rep_features_real.csv"   # or rep_features.csv for synthetic
SEQ_DATA_PATH = "../data/processed/real_sequences.npz"   # set to None to use pseudo-sequences
TEST_SIZE = 0.20
CNN_EPOCHS = 40

MODELS_DIR = Path("../models")
REPORTS_DIR = Path("../outputs/reports")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def cross_validated_recall(model_fn, X, y, n_splits=5):
    """Stratified k-fold CV, reporting recall on the unsafe (positive) class --
    the metric the dissertation's success criterion (recall >= 0.85) is defined on."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    for train_idx, val_idx in skf.split(X, y):
        model = model_fn()
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        scores.append(recall_score(y[val_idx], preds, pos_label=1))
    return float(np.mean(scores)), float(np.std(scores))


def _reconstruct_pseudo_sequence(row: pd.Series, n_steps: int = 30, seed: int = 0) -> np.ndarray:
    """Fallback ONLY -- reconstructs an approximate per-frame sequence from a rep's
    mean/std/range summary stats when no real per-frame sequence data is available
    (i.e. the synthetic dataset). Prefer real sequences from 02_extract_real_dataset.ipynb."""
    rng = np.random.default_rng(seed)
    bases = ["knee_angle_L", "knee_angle_R", "knee_angle_mean",
             "hip_hinge_angle", "trunk_lean_angle", "knee_symmetry"]
    t = np.linspace(0, np.pi, n_steps)
    seq = np.zeros((n_steps, len(bases)))
    for i, base in enumerate(bases):
        mean, std, rng_val = row[f"{base}_mean"], row[f"{base}_std"], row[f"{base}_range"]
        shape = -np.sin(t)
        signal = mean + (rng_val / 2.0) * shape + rng.normal(0, max(std * 0.3, 1e-3), n_steps)
        seq[:, i] = signal
    return seq


def build_cnn_bilstm(n_steps: int, n_features: int):
    from tensorflow import keras
    model = keras.Sequential([
        keras.layers.Input(shape=(n_steps, n_features)),
        keras.layers.Conv1D(32, 3, activation="relu", padding="same"),
        keras.layers.Conv1D(16, 3, activation="relu", padding="same"),
        keras.layers.Bidirectional(keras.layers.LSTM(32, dropout=0.3, recurrent_dropout=0.3)),
        keras.layers.Dense(16, activation="relu"),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy",
                  metrics=["accuracy", keras.metrics.Recall(name="recall")])
    return model

In [ ]:
df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLUMNS].values
y = df["label"].values
print(f"Loaded {len(df)} reps | class balance: {np.bincount(y) / len(y)}")

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index.values, test_size=TEST_SIZE, stratify=y, random_state=42
)

scaler = StandardScaler().fit(X_train)
X_train_s, X_test_s = scaler.transform(X_train), scaler.transform(X_test)
joblib.dump(scaler, MODELS_DIR / "scaler.pkl")

results = {}

In [ ]:
print("[1/4] Logistic Regression")
cv_recall, cv_std = cross_validated_recall(
    lambda: LogisticRegression(max_iter=2000, class_weight="balanced"), X_train_s, y_train)
lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_train_s, y_train)
joblib.dump(lr, MODELS_DIR / "logistic_regression.pkl")
results["logistic_regression"] = {"cv_recall_mean": cv_recall, "cv_recall_std": cv_std}
print(f"  5-fold CV recall (unsafe class): {cv_recall:.3f} +/- {cv_std:.3f}")

In [ ]:
print("[2/4] Random Forest")
cv_recall, cv_std = cross_validated_recall(
    lambda: RandomForestClassifier(n_estimators=300, max_depth=8,
                                    class_weight="balanced", random_state=42),
    X_train, y_train)  # tree models don't need scaling
rf = RandomForestClassifier(n_estimators=300, max_depth=8,
                             class_weight="balanced", random_state=42).fit(X_train, y_train)
joblib.dump(rf, MODELS_DIR / "random_forest.pkl")
results["random_forest"] = {"cv_recall_mean": cv_recall, "cv_recall_std": cv_std}
print(f"  5-fold CV recall (unsafe class): {cv_recall:.3f} +/- {cv_std:.3f}")

In [ ]:
print("[3/4] XGBoost")
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
cv_recall, cv_std = cross_validated_recall(
    lambda: XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                           scale_pos_weight=scale_pos_weight, eval_metric="logloss",
                           random_state=42),
    X_train, y_train)
xgb = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                     scale_pos_weight=scale_pos_weight, eval_metric="logloss",
                     random_state=42).fit(X_train, y_train)
joblib.dump(xgb, MODELS_DIR / "xgboost.pkl")
results["xgboost"] = {"cv_recall_mean": cv_recall, "cv_recall_std": cv_std}
print(f"  5-fold CV recall (unsafe class): {cv_recall:.3f} +/- {cv_std:.3f}")

In [ ]:
n_steps, bases = 30, ["knee_angle_L", "knee_angle_R", "knee_angle_mean",
                       "hip_hinge_angle", "trunk_lean_angle", "knee_symmetry"]

if SEQ_DATA_PATH and Path(SEQ_DATA_PATH).exists():
    print(f"[4/4] CNN + BiLSTM (trained on REAL per-frame sequences from {SEQ_DATA_PATH})")
    seqs = np.load(SEQ_DATA_PATH)["sequences"]
    assert len(seqs) == len(df), (
        f"--seq_data has {len(seqs)} sequences but data has {len(df)} rows -- "
        "must be from the same extraction run.")
else:
    print("[4/4] CNN + BiLSTM (trained on reconstructed pseudo-sequences -- "
          "set SEQ_DATA_PATH for real per-frame sequences)")
    seqs = np.stack([_reconstruct_pseudo_sequence(df.iloc[i], n_steps, seed=i) for i in df.index])

seq_train, seq_test = seqs[idx_train], seqs[idx_test]

seq_mean = seq_train.reshape(-1, seq_train.shape[-1]).mean(axis=0)
seq_std = seq_train.reshape(-1, seq_train.shape[-1]).std(axis=0) + 1e-8
seq_train_n = (seq_train - seq_mean) / seq_std
seq_test_n = (seq_test - seq_mean) / seq_std
np.savez(MODELS_DIR / "cnn_bilstm_scaler.npz", mean=seq_mean, std=seq_std)

cnn = build_cnn_bilstm(n_steps, len(bases))
from tensorflow import keras
early_stop = keras.callbacks.EarlyStopping(monitor="val_recall", mode="max",
                                            patience=8, restore_best_weights=True)
class_weight = {0: 1.0, 1: float((y_train == 0).sum() / max((y_train == 1).sum(), 1))}
history = cnn.fit(seq_train_n, y_train, validation_split=0.2, epochs=CNN_EPOCHS,
                   batch_size=16, class_weight=class_weight, callbacks=[early_stop], verbose=0)
cnn.save(MODELS_DIR / "cnn_bilstm.keras")

val_recall_hist = history.history.get("val_recall", [0])
results["cnn_bilstm"] = {
    "cv_recall_mean": float(np.max(val_recall_hist)),
    "cv_recall_std": 0.0,
    "note": "validation-split recall (held-out fold), not k-fold CV -- see 04_evaluate.ipynb "
            "for the proper held-out test-set comparison across all 4 models.",
}
print(f"  best validation recall (unsafe class): {results['cnn_bilstm']['cv_recall_mean']:.3f}")

In [ ]:
np.savez(MODELS_DIR / "test_split.npz",
          X_test=X_test, X_test_s=X_test_s, y_test=y_test,
          seq_test_n=seq_test_n, idx_test=idx_test)

with open(REPORTS_DIR / "cv_training_summary.json", "w") as f:
    json.dump(results, f, indent=2)

print("Training complete. Cross-validation recall summary:")
for name, r in results.items():
    print(f"  {name:22s} recall={r['cv_recall_mean']:.3f}")
print(f"\nModels saved to {MODELS_DIR}")
print("Run 04_evaluate.ipynb next for held-out test metrics, ROC curves, and McNemar's test.")